# Generate SPFC target-only uniform with unclipped VFA on T2I-CompBench indexes 50 to 99 (Local)

Runs only `spfc_target_only_uniform_vfa_unclipped` for the end-exclusive manifest slice `50:100`.

Outputs are written under the repo root's `t2i_compbench_seed13/runs/t2i_compbench/spfc_target_only_uniform_vfa_unclipped` directory.

In [1]:
from pathlib import Path


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for path in (start, *start.parents):
        if (path / "src" / "aim_flow").exists() and (path / "scripts" / "bench_generate.py").exists():
            return path
    raise RuntimeError("Could not find the aim-flow repo root from the current notebook location.")


REPO_DIR = find_repo_root()
%cd {REPO_DIR}
print(f"repo: {REPO_DIR}")

/home/cvgl/spfc/aim-flow
repo: /home/cvgl/spfc/aim-flow


/home/cvgl/spfc/aim-flow/.venv/t2i-compbench-py310/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [2]:
from pathlib import Path
import json
import os
import shutil
import shlex
import subprocess
import sys
import time

os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
os.environ.setdefault("TRANSFORMERS_VERBOSITY", "error")
os.environ.setdefault("DIFFUSERS_VERBOSITY", "error")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("PYTHONUNBUFFERED", "1")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

SEED = 13
RUN_SLUG = "t2i_compbench_seed13"
WORK_ROOT = Path("/kaggle/working") if Path("/kaggle").exists() else Path.cwd()
OUTPUT_ROOT = WORK_ROOT / RUN_SLUG
RUN_ROOT = OUTPUT_ROOT / "runs"
MANIFEST_REL = Path("configs/t2i_compbench_100_seed13.json")
DECOMP_REL = Path("configs/t2i_compbench_100_seed13_spfc.json")
CONFIG_REL = Path("configs/sd3_medium_kaggle_vfa_unclipped.yaml")
RECTIFIED_REPO_DIR = WORK_ROOT / "Rectified-CFGpp"
INSTALL_DEPS = True
GPU_SELECTION = "2"
SHARD_START = 50
SHARD_END = 100
EXPECTED_SHARD_COUNT = SHARD_END - SHARD_START
SHARD_LABEL = "050-099"

# If your repo is attached with a different Kaggle dataset slug, this auto-discovers it under /kaggle/input.
def has_aim_flow_repo(path: Path) -> bool:
    return (path / "src" / "aim_flow").exists() and (path / "scripts" / "bench_generate.py").exists()


def find_aim_flow_repo() -> Path | None:
    cwd = Path.cwd()
    if has_aim_flow_repo(cwd):
        return cwd
    dest = WORK_ROOT / "aim-flow"
    if has_aim_flow_repo(dest):
        return dest
    input_root = Path("/kaggle/input")
    candidates = [input_root / "aim-flow", input_root / "aim-flow" / "aim-flow"]
    if input_root.exists():
        for child in sorted(input_root.glob("*")):
            candidates.extend([child, child / "aim-flow"])
    for candidate in candidates:
        if has_aim_flow_repo(candidate):
            return candidate
    return None


def ensure_working_repo() -> Path:
    source = find_aim_flow_repo()
    if source is None:
        raise FileNotFoundError(
            "Could not find aim-flow. Attach the repo as a Kaggle dataset, clone it into /kaggle/working, "
            "or run this notebook from the repo root."
        )
    dest = WORK_ROOT / "aim-flow" if Path("/kaggle").exists() else source
    if source.resolve() != dest.resolve():
        shutil.copytree(source, dest, dirs_exist_ok=True)
        return dest
    return source


def run_args(args: list[str], cwd: Path | None = None, check: bool = True) -> subprocess.CompletedProcess:
    print("$", shlex.join([str(arg) for arg in args]))
    started = time.time()
    result = subprocess.run([str(arg) for arg in args], cwd=str(cwd) if cwd else None, text=True)
    print(f"elapsed: {(time.time() - started) / 60:.2f} min")
    if check and result.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {result.returncode}")
    return result


def normalize_gpu_selection(value: str) -> str:
    selection = ",".join(part.strip() for part in str(value).split(",") if part.strip())
    if not selection:
        raise ValueError("GPU_SELECTION must be a non-empty string like '0' or '0,1'.")
    if any(not part.isdigit() for part in selection.split(",")):
        raise ValueError(f"GPU_SELECTION must contain only GPU indices, got: {value!r}")
    return selection


def torch_runtime_ready() -> bool:
    probe = "import torch\nimport torch.nn.functional as F\nprint(torch.__version__)"
    result = subprocess.run([str(RUN_PYTHON), "-c", probe], cwd=str(REPO_DIR), text=True)
    return result.returncode == 0


REPO_DIR = ensure_working_repo()
os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR / "src"))
ENV_PREFIX = REPO_DIR / ".venv" / "t2i-compbench-py310"
RUN_PYTHON = (ENV_PREFIX / "bin" / "python").resolve()
KERNEL_PYTHON = Path(sys.executable).resolve()
if not RUN_PYTHON.exists():
    raise FileNotFoundError(f"Expected local evaluator venv at {RUN_PYTHON}")
if KERNEL_PYTHON != RUN_PYTHON:
    raise RuntimeError(
        f"This local notebook must use the repo venv kernel at {RUN_PYTHON}. Current kernel is {KERNEL_PYTHON}. "
        "Select that interpreter in VS Code, then rerun the notebook from the top."
    )
GPU_SELECTION = normalize_gpu_selection(GPU_SELECTION)
os.environ["CUDA_VISIBLE_DEVICES"] = GPU_SELECTION
if not torch_runtime_ready():
    run_args(
        [
            RUN_PYTHON,
            "-m",
            "pip",
            "install",
            "-q",
            "--no-cache-dir",
            "--force-reinstall",
            "torch==2.4.1",
            "torchvision==0.19.1",
        ],
        cwd=REPO_DIR,
    )
    if not torch_runtime_ready():
        raise RuntimeError(f"Torch is still unavailable in {RUN_PYTHON} after reinstall.")
print("repo:", REPO_DIR)
print("python:", RUN_PYTHON)
print("CUDA_VISIBLE_DEVICES:", os.environ["CUDA_VISIBLE_DEVICES"])
print("outputs:", OUTPUT_ROOT)
print("config:", REPO_DIR / CONFIG_REL)

if INSTALL_DEPS:
    run_args([RUN_PYTHON, "-m", "pip", "install", "-q", "-r", "requirements.txt"], cwd=REPO_DIR)
    run_args([RUN_PYTHON, "-m", "pip", "install", "-q", "--no-deps", "-e", "."], cwd=REPO_DIR)

2.12.0+cu130
repo: /home/cvgl/spfc/aim-flow
python: /home/cvgl/spfc/aim-flow/.venv/t2i-compbench-py310/bin/python3.10
CUDA_VISIBLE_DEVICES: 2
outputs: /home/cvgl/spfc/aim-flow/t2i_compbench_seed13
config: /home/cvgl/spfc/aim-flow/configs/sd3_medium_kaggle_vfa_unclipped.yaml
$ /home/cvgl/spfc/aim-flow/.venv/t2i-compbench-py310/bin/python3.10 -m pip install -q -r requirements.txt
elapsed: 0.02 min
$ /home/cvgl/spfc/aim-flow/.venv/t2i-compbench-py310/bin/python3.10 -m pip install -q --no-deps -e .
elapsed: 0.06 min


In [3]:
METHOD_LABEL = "spfc_target_only_uniform_vfa_unclipped"
METHOD_TITLE = "SPFC target-only uniform with unclipped VFA"
SPFC_VARIANT = "target_only_uniform"
GUIDANCE_SCALE = 4.5
print(f"Generating {METHOD_TITLE} shard {SHARD_LABEL}: indexes {SHARD_START} to {SHARD_END - 1}")

Generating SPFC target-only uniform with unclipped VFA shard 050-099: indexes 50 to 99


In [4]:
import os
from IPython.display import Markdown, display

MODEL_ID = 'stabilityai/stable-diffusion-3-medium-diffusers'
ENV_PATH = REPO_DIR / '.env'
TOKEN_ENV_KEYS = ('HF_TOKEN', 'HUGGINGFACE_TOKEN')

if ENV_PATH.exists():
    for raw_line in ENV_PATH.read_text(encoding='utf-8').splitlines():
        line = raw_line.strip()
        if not line or line.startswith('#') or '=' not in line:
            continue
        key, value = line.split('=', 1)
        key = key.strip()
        value = value.strip().strip('"').strip("'")
        if key in TOKEN_ENV_KEYS and value:
            os.environ.setdefault(key, value)

token = os.environ.get('HF_TOKEN') or os.environ.get('HUGGINGFACE_TOKEN')
if not token:
    raise RuntimeError(
        f'Missing Hugging Face token. Add HF_TOKEN to {ENV_PATH} and make sure that account has accepted the SD3 Medium license.'
    )

try:
    from huggingface_hub import HfApi
    HfApi().model_info(MODEL_ID, token=token)
except Exception as exc:
    raise RuntimeError(
        f'HF token is set, but access check for {MODEL_ID} failed. Confirm the token in {ENV_PATH} is valid and that the account has accepted the gated model license.'
    ) from exc

display(Markdown(f'Hugging Face token loaded from {ENV_PATH} and can access SD3 Medium.'))

run_args(["nvidia-smi"], check=False)

/home/cvgl/spfc/aim-flow/.venv/t2i-compbench-py310/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Hugging Face token loaded from /home/cvgl/spfc/aim-flow/.env and can access SD3 Medium.

$ nvidia-smi
Sun May 31 05:01:56 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.03             Driver Version: 580.159.03     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 2080 Ti     Off |   00000000:17:00.0 Off |                  N/A |
| 46%   55C    P2            262W /  260W |    8955MiB /  11264MiB |     99%      Default |
|                                         |                        |                  N/A |
+----------------------------------

CompletedProcess(args=['nvidia-smi'], returncode=0)

In [5]:
manifest_path = REPO_DIR / MANIFEST_REL
decomp_path = REPO_DIR / DECOMP_REL
if not manifest_path.exists():
    raise FileNotFoundError(manifest_path)
if not decomp_path.exists():
    raise FileNotFoundError(decomp_path)

with manifest_path.open("r", encoding="utf-8") as f:
    manifest = json.load(f)
with decomp_path.open("r", encoding="utf-8") as f:
    decompositions = json.load(f)

sample_count = len(manifest["samples"])
decomposition_count = len(decompositions["items"])
expected_ids = [sample["id"] for sample in manifest["samples"][SHARD_START:SHARD_END]]
print("manifest:", manifest_path)
print("decompositions:", decomp_path)
print("benchmark:", manifest["benchmark"])
print("full samples:", sample_count)
print("shard samples:", len(expected_ids))
assert sample_count == 100, f"Expected 100 T2I-CompBench samples, found {sample_count}."
assert decomposition_count == 100, f"Expected 100 SPFC decompositions, found {decomposition_count}."
assert len(expected_ids) == EXPECTED_SHARD_COUNT == 50

from aim_flow.eval_bench.generation import apply_spfc_variant, load_bench_config

target_only_config = apply_spfc_variant(
    load_bench_config(config_path=REPO_DIR / CONFIG_REL, seed=SEED, guidance_scale=GUIDANCE_SCALE),
    SPFC_VARIANT,
)
primitive_flow = target_only_config.primitive_flow
assert primitive_flow.uniform_condition_weights is True
assert primitive_flow.use_consensus_gating is False
assert primitive_flow.use_target_consistency_gating is True
assert primitive_flow.source_weight == 1.0
assert primitive_flow.target_weight == 1.0
assert primitive_flow.velocity_clip_ratio == 1000000.0
print("validated SPFC target-only-uniform settings with unclipped VFA")

manifest: /home/cvgl/spfc/aim-flow/configs/t2i_compbench_100_seed13.json
decompositions: /home/cvgl/spfc/aim-flow/configs/t2i_compbench_100_seed13_spfc.json
benchmark: t2i_compbench
full samples: 100
shard samples: 50
validated SPFC target-only-uniform settings with unclipped VFA


In [6]:
cmd = [
    RUN_PYTHON,
    "scripts/bench_generate.py",
    "--manifest",
    REPO_DIR / MANIFEST_REL,
    "--run-root",
    RUN_ROOT,
    "--methods",
    "spfc",
    "--decompositions",
    REPO_DIR / DECOMP_REL,
    "--config",
    REPO_DIR / CONFIG_REL,
    "--seed",
    str(SEED),
    "--guidance-scale",
    str(GUIDANCE_SCALE),
    "--sample-start",
    str(SHARD_START),
    "--sample-end",
    str(SHARD_END),
    "--spfc-variant",
    SPFC_VARIANT,
    "--spfc-method-label",
    METHOD_LABEL,
    "--skip-existing",
]
run_args(cmd, cwd=REPO_DIR)

$ /home/cvgl/spfc/aim-flow/.venv/t2i-compbench-py310/bin/python3.10 scripts/bench_generate.py --manifest /home/cvgl/spfc/aim-flow/configs/t2i_compbench_100_seed13.json --run-root /home/cvgl/spfc/aim-flow/t2i_compbench_seed13/runs --methods spfc --decompositions /home/cvgl/spfc/aim-flow/configs/t2i_compbench_100_seed13_spfc.json --config /home/cvgl/spfc/aim-flow/configs/sd3_medium_kaggle_vfa_unclipped.yaml --seed 13 --guidance-scale 4.5 --sample-start 50 --sample-end 100 --spfc-variant target_only_uniform --spfc-method-label spfc_target_only_uniform_vfa_unclipped --skip-existing


Loading pipeline components...: 100%|██████████| 7/7 [00:01<00:00,  3.62it/s]
t2i_compbench/spfc_target_only_uniform_vfa_unclipped image 1/50:   0%|          | 0/50 [00:00<?, ?image/s, left=49]
PrimitiveFlow primitive_flow_sparse: 100%|██████████| 24/24 [00:16<00:00,  1.49it/s]
t2i_compbench/spfc_target_only_uniform_vfa_unclipped image 2/50:   2%|▏         | 1/50 [00:26<21:36, 26.45s/image, left=48]
PrimitiveFlow primitive_flow_sparse: 100%|██████████| 24/24 [00:15<00:00,  1.51it/s]
t2i_compbench/spfc_target_only_uniform_vfa_unclipped image 3/50:   4%|▍         | 2/50 [00:49<19:39, 24.57s/image, left=47]
PrimitiveFlow primitive_flow_sparse: 100%|██████████| 24/24 [00:16<00:00,  1.50it/s]
t2i_compbench/spfc_target_only_uniform_vfa_unclipped image 4/50:   6%|▌         | 3/50 [01:11<18:24, 23.50s/image, left=46]
PrimitiveFlow primitive_flow_sparse: 100%|██████████| 24/24 [00:16<00:00,  1.50it/s]
t2i_compbench/spfc_target_only_uniform_vfa_unclipped image 5/50:   8%|▊         | 4/50 [01:34<

generation_indices:
  spfc_target_only_uniform_vfa_unclipped: /home/cvgl/spfc/aim-flow/t2i_compbench_seed13/runs/t2i_compbench/spfc_target_only_uniform_vfa_unclipped/index.json
elapsed: 20.38 min


CompletedProcess(args=['/home/cvgl/spfc/aim-flow/.venv/t2i-compbench-py310/bin/python3.10', 'scripts/bench_generate.py', '--manifest', '/home/cvgl/spfc/aim-flow/configs/t2i_compbench_100_seed13.json', '--run-root', '/home/cvgl/spfc/aim-flow/t2i_compbench_seed13/runs', '--methods', 'spfc', '--decompositions', '/home/cvgl/spfc/aim-flow/configs/t2i_compbench_100_seed13_spfc.json', '--config', '/home/cvgl/spfc/aim-flow/configs/sd3_medium_kaggle_vfa_unclipped.yaml', '--seed', '13', '--guidance-scale', '4.5', '--sample-start', '50', '--sample-end', '100', '--spfc-variant', 'target_only_uniform', '--spfc-method-label', 'spfc_target_only_uniform_vfa_unclipped', '--skip-existing'], returncode=0)

In [7]:
method_dir = RUN_ROOT / "t2i_compbench" / METHOD_LABEL
index_path = method_dir / "index.json"
if not index_path.exists():
    raise FileNotFoundError(f"Missing generation index: {index_path}")
with index_path.open("r", encoding="utf-8") as f:
    index = json.load(f)
output_ids = [item["sample_id"] for item in index["outputs"]]
assert output_ids == expected_ids, f"Unexpected output IDs for {METHOD_LABEL}"
print(f"{METHOD_LABEL}: {len(output_ids)} outputs at {method_dir}")

readme = OUTPUT_ROOT / f"README_{METHOD_LABEL}_shard_{SHARD_LABEL}.txt"
readme.write_text(
    "T2I-CompBench SPFC shard\n"
    f"method: {METHOD_LABEL}\n"
    f"variant: {SPFC_VARIANT or 'full'}\n"
    f"indexes: {SHARD_START} to {SHARD_END - 1}\n"
    f"slice: {SHARD_START}:{SHARD_END}\n"
    f"seed: {SEED}\n"
    f"guidance_scale: {GUIDANCE_SCALE}\n"
    f"method_dir: {method_dir}\n",
    encoding="utf-8",
)
print("index:", index_path)
print("readme:", readme)
print("Kaggle output root:", OUTPUT_ROOT)
print("Merge this notebook output with the matching method's other shard before the comparison run.")

spfc_target_only_uniform_vfa_unclipped: 50 outputs at /home/cvgl/spfc/aim-flow/t2i_compbench_seed13/runs/t2i_compbench/spfc_target_only_uniform_vfa_unclipped
index: /home/cvgl/spfc/aim-flow/t2i_compbench_seed13/runs/t2i_compbench/spfc_target_only_uniform_vfa_unclipped/index.json
readme: /home/cvgl/spfc/aim-flow/t2i_compbench_seed13/README_spfc_target_only_uniform_vfa_unclipped_shard_050-099.txt
Kaggle output root: /home/cvgl/spfc/aim-flow/t2i_compbench_seed13
Merge this notebook output with the matching method's other shard before the comparison run.
